In [180]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest

### We will be using isolation forest to detect the anamolies

In [181]:
data = pd.read_parquet('../updatedtop20datasets/META.parquet')
data.head(5)

,date,close,high,low,open,volume,adjClose,adjHigh,adjLow,adjOpen,adjVolume,divCash,splitFactor,Symbol,price_return,Volatility_7Days,Volatility_30Days,Price_Swing,volume_zscore7Days,volume_zscore30Days,MASignal,year,volatility_diff,volume_z_diff,is_anomaly,price_score,volume_score,volatility_score,cum_score
0,2012-05-18 00:00:00+00:00,38.2318,45.00,38.00,42.05,573576400,37.937002,44.653014,37.706990,41.725761,573576400,0.0,1.0,META,0.000000,0.0,0.0,0.183094,0.0,0.0,0,2012,0.0,0.0,0,-0.103761,0.181175,0.133157,0.070190
1,2012-05-21 00:00:00+00:00,34.0300,36.66,33.00,36.53,168192700,33.767602,36.377322,32.745544,36.248325,168192700,0.0,1.0,META,-0.109903,0.0,0.0,0.107552,0.0,0.0,0,2012,0.0,0.0,1,-0.149288,0.181175,0.133157,0.055015
2,2012-05-22 00:00:00+00:00,31.0000,33.59,30.94,32.61,101786600,30.760965,33.330994,30.701428,32.358551,101786600,0.0,1.0,META,-0.089039,0.0,0.0,0.085484,0.0,0.0,0,2012,0.0,0.0,0,-0.119364,0.181175,0.133157,0.064989
3,2012-05-23 00:00:00+00:00,32.0000,32.50,31.36,31.37,73600000,31.753255,32.249399,31.118189,31.128112,73600000,0.0,1.0,META,0.032258,0.0,0.0,0.035625,0.0,0.0,0,2012,0.0,0.0,0,0.102737,0.181175,0.133157,0.139023
4,2012-05-24 00:00:00+00:00,33.0300,33.21,31.77,32.95,50237200,32.775312,32.953925,31.525028,32.695929,50237200,0.0,1.0,META,0.032187,0.0,0.0,0.043597,0.0,0.0,0,2012,0.0,0.0,0,0.079452,0.181175,0.133157,0.131261


In [189]:
def data_partition(symbol):
    path = '../updatedtop20datasets/'
    filepath = f'{path}{symbol}.parquet'
    columns =  ['price_return', 'volatility_diff', 'Volatility_30Days', 'Price_Swing', 'volume_z_diff', 'volume_zscore30Days']
    data = pd.read_parquet(filepath)
    scaler = StandardScaler()
    data[columns] = scaler.fit_transform(data[columns])
    data.to_parquet(filepath, index = False)
    return data

In [183]:
def iso_runner(symbol,contamination = 0.05):
    path = '../updatedtop20datasets/'
    filepath = f'{path}{symbol}.parquet'
    data = pd.read_parquet(filepath)
    price_features = ['price_return', 'Price_Swing', 'MASignal']
    volume_features = ['volume_z_diff', 'volume_zscore30Days']
    volatility_features = ['volatility_diff', 'Volatility_30Days']
    
    for features, col_name in zip(
        [price_features, volume_features, volatility_features],
        ['price_score', 'volume_score', 'volatility_score']
    ):
        model = IsolationForest(contamination=contamination, random_state=42)
        model.fit(data[features])
        data[col_name] = model.decision_function(data[features])
    
    data['cum_score'] = (data['price_score'] + 
                         data['volume_score'] + 
                         data['volatility_score']) / 3
    data['is_anomaly'] = (data['cum_score'] < data['cum_score'].quantile(0.05)).astype(int)
    data.to_parquet(filepath, index=False)
    return data

print(data['is_anomaly'].value_counts())

is_anomaly
0    3350
1     177
Name: count, dtype: int64


In [184]:
data.head(5)

,date,close,high,low,open,volume,adjClose,adjHigh,adjLow,adjOpen,adjVolume,divCash,splitFactor,Symbol,price_return,Volatility_7Days,Volatility_30Days,Price_Swing,volume_zscore7Days,volume_zscore30Days,MASignal,year,volatility_diff,volume_z_diff,is_anomaly,price_score,volume_score,volatility_score,cum_score
0,2012-05-18 00:00:00+00:00,38.2318,45.00,38.00,42.05,573576400,37.937002,44.653014,37.706990,41.725761,573576400,0.0,1.0,META,0.000000,0.0,0.0,0.183094,0.0,0.0,0,2012,0.0,0.0,0,-0.103761,0.181175,0.133157,0.070190
1,2012-05-21 00:00:00+00:00,34.0300,36.66,33.00,36.53,168192700,33.767602,36.377322,32.745544,36.248325,168192700,0.0,1.0,META,-0.109903,0.0,0.0,0.107552,0.0,0.0,0,2012,0.0,0.0,1,-0.149288,0.181175,0.133157,0.055015
2,2012-05-22 00:00:00+00:00,31.0000,33.59,30.94,32.61,101786600,30.760965,33.330994,30.701428,32.358551,101786600,0.0,1.0,META,-0.089039,0.0,0.0,0.085484,0.0,0.0,0,2012,0.0,0.0,0,-0.119364,0.181175,0.133157,0.064989
3,2012-05-23 00:00:00+00:00,32.0000,32.50,31.36,31.37,73600000,31.753255,32.249399,31.118189,31.128112,73600000,0.0,1.0,META,0.032258,0.0,0.0,0.035625,0.0,0.0,0,2012,0.0,0.0,0,0.102737,0.181175,0.133157,0.139023
4,2012-05-24 00:00:00+00:00,33.0300,33.21,31.77,32.95,50237200,32.775312,32.953925,31.525028,32.695929,50237200,0.0,1.0,META,0.032187,0.0,0.0,0.043597,0.0,0.0,0,2012,0.0,0.0,0,0.079452,0.181175,0.133157,0.131261


In [185]:
anomaly = data[data['is_anomaly'] == 1]
anomaly.describe()

,close,high,low,open,volume,adjClose,adjHigh,adjLow,adjOpen,adjVolume,divCash,splitFactor,price_return,Volatility_7Days,Volatility_30Days,Price_Swing,volume_zscore7Days,volume_zscore30Days,MASignal,year,volatility_diff,volume_z_diff,is_anomaly,price_score,volume_score,volatility_score,cum_score
count,177.000000,177.000000,177.000000,177.000000,1.770000e+02,177.000000,177.000000,177.000000,177.000000,1.770000e+02,177.0,177.0,177.000000,177.000000,177.000000,177.000000,177.000000,177.000000,177.000000,177.000000,177.000000,177.000000,177.0,177.000000,177.000000,177.000000,177.000000
mean,322.967994,330.452042,316.047576,323.634615,5.963746e+07,321.561153,329.008096,314.673926,322.224170,5.963746e+07,0.0,1.0,0.000772,12.888756,23.220390,0.054240,1.043901,1.816390,0.361582,2021.079096,-10.331634,-0.772489,1.0,-0.002619,0.007183,0.069350,0.024638
std,225.983130,229.402158,222.477703,226.158838,5.205835e+07,225.639448,229.054470,222.136724,225.815734,5.205835e+07,0.0,0.0,0.075033,11.525784,18.259041,0.027080,1.160956,1.764156,0.481822,4.213677,17.631413,1.280602,0.0,0.094851,0.086331,0.113972,0.035376
min,19.870000,20.480000,19.690000,20.100000,6.743473e+06,19.716787,20.322083,19.538174,19.945013,6.743473e+06,0.0,1.0,-0.263901,0.000000,0.000000,0.010439,-1.921355,-1.585077,0.000000,2012.000000,-50.800934,-2.851800,1.0,-0.184573,-0.177292,-0.146133,-0.109593
25%,149.730000,159.270000,144.800000,152.320000,2.709490e+07,148.575463,158.041902,143.683477,151.145492,2.709490e+07,0.0,1.0,-0.044736,5.322097,7.045384,0.035899,0.259318,0.144429,0.000000,2019.000000,-21.648864,-1.594441,1.0,-0.080915,-0.036876,-0.027325,0.012157
50%,220.640000,230.420000,216.150000,223.500000,4.285959e+07,218.938690,228.643279,214.483312,221.776637,4.285959e+07,0.0,1.0,-0.007408,8.940568,17.493662,0.049780,1.433748,1.752961,0.000000,2022.000000,-2.916222,-1.142273,1.0,-0.014983,0.014922,0.079105,0.033393
75%,572.130000,583.000000,546.770000,581.395000,7.664563e+07,570.450195,583.000000,546.770000,578.643089,7.664563e+07,0.0,1.0,0.045644,16.318204,39.188537,0.072134,2.015624,3.399619,1.000000,2025.000000,1.391497,0.076032,1.0,0.067696,0.065122,0.166286,0.050725
max,773.440000,784.750000,765.510000,775.200000,3.654579e+08,771.637872,782.921519,763.726349,773.393771,3.654579e+08,0.0,1.0,0.296077,57.965498,59.349408,0.184432,2.260472,5.086715,1.000000,2026.000000,24.856223,2.289011,1.0,0.205735,0.187427,0.249996,0.061205


In [186]:
non_anomaly = data[data['is_anomaly'] == 0]
non_anomaly.describe()

,close,high,low,open,volume,adjClose,adjHigh,adjLow,adjOpen,adjVolume,divCash,splitFactor,price_return,Volatility_7Days,Volatility_30Days,Price_Swing,volume_zscore7Days,volume_zscore30Days,MASignal,year,volatility_diff,volume_z_diff,is_anomaly,price_score,volume_score,volatility_score,cum_score
count,3350.000000,3350.000000,3350.000000,3350.000000,3.350000e+03,3350.000000,3350.000000,3350.000000,3350.000000,3.350000e+03,3350.000000,3350.0,3350.000000,3350.000000,3350.000000,3350.000000,3350.000000,3350.000000,3350.000000,3350.000000,3350.000000,3350.000000,3350.0,3350.000000,3350.000000,3350.000000,3350.000000
mean,228.161823,230.841078,225.348138,228.096208,2.625424e+07,226.815617,229.478982,224.018748,226.750677,2.625424e+07,0.001381,1.0,0.001128,4.577938,9.448213,0.025039,-0.087126,-0.103819,0.640000,2018.773731,-4.870275,0.016693,0.0,0.142678,0.134496,0.192151,0.156442
std,186.058896,188.195979,183.866151,186.146264,2.138120e+07,185.564964,187.696222,183.378693,185.652592,2.138120e+07,0.026612,0.0,0.019040,5.077478,9.616784,0.012644,0.950121,0.903006,0.480072,4.025838,6.972026,0.689279,0.0,0.053401,0.057549,0.065179,0.034836
min,17.729000,18.270000,17.550000,18.080000,4.726056e+06,17.592295,18.129124,17.414676,17.940589,4.726056e+06,0.000000,1.0,-0.090551,0.000000,0.000000,0.003715,-2.154895,-2.416273,0.000000,2012.000000,-43.669148,-2.555550,0.0,-0.159596,-0.153764,-0.112156,0.061800
25%,96.532500,97.632500,94.832500,96.440000,1.436644e+07,95.788158,96.879676,94.101266,95.696371,1.436644e+07,0.000000,1.0,-0.009510,1.321202,2.683579,0.016241,-0.777292,-0.672795,0.000000,2015.000000,-6.378278,-0.451423,0.0,0.118152,0.107113,0.181418,0.133260
50%,174.240000,176.175000,172.090000,174.200000,2.012835e+07,172.896471,174.816551,170.763049,172.856780,2.012835e+07,0.000000,1.0,0.001074,2.871522,6.163604,0.022535,-0.310963,-0.308884,1.000000,2019.000000,-2.578878,-0.023006,0.0,0.155303,0.151473,0.214793,0.164490
75%,298.887500,302.542500,295.250000,298.921250,3.027208e+07,296.582840,300.209657,292.973388,296.616330,3.027208e+07,0.000000,1.0,0.012362,5.961363,12.026361,0.031122,0.500420,0.218153,1.000000,2022.000000,-0.827470,0.420020,0.0,0.184230,0.179902,0.233780,0.183692
max,790.000000,796.250000,780.820000,791.150000,5.735764e+08,788.159287,794.394724,779.000676,789.306607,5.735764e+08,0.525000,1.0,0.102493,54.205956,54.222135,0.183094,2.256750,4.729619,1.000000,2026.000000,18.860092,2.532857,0.0,0.209483,0.204973,0.253422,0.216120


## For all the 20 datas we have ! 


In [187]:
### Creating datasets for the top 20 companies 
watchlist = {
    "AAPL":  "Apple",
    "MSFT":  "Microsoft", 
    "NVDA":  "Nvidia",
    "GOOGL": "Alphabet (Google)",
    "AMD":  "AMD",
    "META":  "Meta (Facebook)",
    "TSLA":  "Tesla",
    "NFLX": "Netflix",
    "LLY":   "Eli Lilly",
    "AVGO":   "Broadcom",
    "MU":    "Micron Technology",
    "QCOM":   "Qualcomm",
    "UNH":   "UnitedHealth",
    "WMT":   "Walmart",
    "MA":    "Mastercard",
    "JNJ":   "Johnson & Johnson",
    "PG":    "Procter & Gamble",
    "HD":    "Home Depot",
    "ORCL":  "Oracle",
    "JPM": "JPMorgan Chase"
}


In [188]:
for key,value in watchlist.items():
    data_partition(key)
features = [['price_return','Price_Swing'], ['volatility_diff', 'Volatility_30Days'], ['volume_z_diff', 'volume_zscore30Days']]
for key,value in watchlist.items():
    iso_runner(key)
    print(f"{key} done")

KeyError: "['MASignal'] not in index"

In [ ]:
data = pd.read_parquet('../updatedtop20datasets/AVGO.parquet')
data.head(5)

,date,close,high,low,open,volume,adjClose,adjHigh,adjLow,adjOpen,adjVolume,divCash,splitFactor,Symbol,price_return,Volatility_7Days,Volatility_30Days,Price_Swing,volume_zscore7Days,volume_zscore30Days,MASignal,year,volatility_diff,volume_z_diff,is_anomaly,price_score,volume_score,volatility_score,cum_score
0,2012-05-18 00:00:00+00:00,38.2318,45.00,38.00,42.05,573576400,37.937002,44.653014,37.706990,41.725761,573576400,0.0,1.0,META,0.000000,0.0,0.0,0.183094,0.0,0.0,0,2012,0.0,0.0,0,-0.103761,0.181175,0.133157,0.070190
1,2012-05-21 00:00:00+00:00,34.0300,36.66,33.00,36.53,168192700,33.767602,36.377322,32.745544,36.248325,168192700,0.0,1.0,META,-0.109903,0.0,0.0,0.107552,0.0,0.0,0,2012,0.0,0.0,1,-0.149288,0.181175,0.133157,0.055015
2,2012-05-22 00:00:00+00:00,31.0000,33.59,30.94,32.61,101786600,30.760965,33.330994,30.701428,32.358551,101786600,0.0,1.0,META,-0.089039,0.0,0.0,0.085484,0.0,0.0,0,2012,0.0,0.0,0,-0.119364,0.181175,0.133157,0.064989
3,2012-05-23 00:00:00+00:00,32.0000,32.50,31.36,31.37,73600000,31.753255,32.249399,31.118189,31.128112,73600000,0.0,1.0,META,0.032258,0.0,0.0,0.035625,0.0,0.0,0,2012,0.0,0.0,0,0.102737,0.181175,0.133157,0.139023
4,2012-05-24 00:00:00+00:00,33.0300,33.21,31.77,32.95,50237200,32.775312,32.953925,31.525028,32.695929,50237200,0.0,1.0,META,0.032187,0.0,0.0,0.043597,0.0,0.0,0,2012,0.0,0.0,0,0.079452,0.181175,0.133157,0.131261
